In [87]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

### 1. Read Data

In [88]:
df = pd.read_csv('../data/raw/us_housing_dataset.csv')
df.head()

,property_id,listing_id,listing_status,list_price,sold_price,beds,baths,sqft_living,lot_sqft,year_built,...,zip_code,latitude,longitude,hoa_fee,days_on_market,price_per_sqft,listing_type,scraped_at,parking_garage,stories
0,8579831246,2992513681,FOR_SALE,1674999,NaN,2.0,2.0,1587.0,131.0,2011.0,...,78701,30.265737,-97.746570,1808.0,149.0,1055.0,for_sale,2026-08-10T08:49:20.483153,1.0,1.0
1,8665330750,2995474038,FOR_SALE,19900000,NaN,3.0,4.0,15600.0,3920.0,1871.0,...,78701,30.270049,-97.741605,0.0,86.0,1276.0,for_sale,2026-08-10T08:49:20.483153,1.0,3.0
2,8538310811,2998333509,FOR_SALE,14950000,NaN,5.0,8.0,8803.0,3681.0,1930.0,...,78701,30.270243,-97.741535,0.0,31.0,1698.0,for_sale,2026-08-10T08:49:20.483153,3.0,2.0
3,8059934237,2997241017,FOR_SALE,1495000,NaN,4.0,3.0,2057.0,144.0,2008.0,...,78701,30.267661,-97.749622,1976.0,58.0,727.0,for_sale,2026-08-10T08:49:20.483153,4.0,1.0
4,8411588968,2994860259,FOR_SALE,495000,NaN,2.0,2.0,1169.0,301.0,1964.0,...,78701,30.280650,-97.740283,1404.0,96.0,423.0,for_sale,2026-08-10T08:49:20.483153,1.0,1.0


### 2. Check Dataset Structure

In [89]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2766 entries, 0 to 2765
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   property_id     2766 non-null   int64  
 1   listing_id      2766 non-null   int64  
 2   listing_status  2766 non-null   object 
 3   list_price      2766 non-null   int64  
 4   sold_price      0 non-null      float64
 5   beds            2742 non-null   float64
 6   baths           2740 non-null   float64
 7   sqft_living     2736 non-null   float64
 8   lot_sqft        2601 non-null   float64
 9   year_built      2739 non-null   float64
 10  property_type   2766 non-null   object 
 11  address         2766 non-null   object 
 12  city            2766 non-null   object 
 13  state           2766 non-null   object 
 14  zip_code        2766 non-null   int64  
 15  latitude        2766 non-null   float64
 16  longitude       2766 non-null   float64
 17  hoa_fee         2640 non-null   f

### 3. Missing Values Handling

In [90]:
# Drop empty columns
cols_to_drop = df.isnull().sum()[df.isnull().sum() == len(df)].index
df = df.drop(columns=cols_to_drop)

In [91]:
# Impute numeric features with median
num_features = df.select_dtypes(include=['number']).columns
df[num_features] = df[num_features].fillna(df[num_features].median())

# Impute categorical features with mode
cat_features = df.select_dtypes(include=['object', 'category']).columns
for col in cat_features:
    df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown')

print('Missing values after imputation:', df.isnull().sum().sum())

Missing values after imputation: 0


### Duplicate Removal
Remove identical or highly duplicated rows.

In [92]:
duplicates_count = df.duplicated().sum()
print(f"Found {duplicates_count} duplicated rows.")
df = df.drop_duplicates()
print(f"Total records remaining: {len(df)}")

Found 0 duplicated rows.
Total records remaining: 2766


### Outlier Detection and Handling
We'll use the IQR (Interquartile Range) method to remove extreme outliers in the dataset.

In [93]:
def remove_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Filtering out the outliers
    df_filtered = dataframe[(dataframe[column] >= lower_bound) & (dataframe[column] <= upper_bound)]
    print(f"Removed {len(dataframe) - len(df_filtered)} outliers based on {column}")
    return df_filtered

original_len = len(df)
df = remove_outliers_iqr(df, 'list_price')
df = remove_outliers_iqr(df, 'sqft_living')

print(f"Total records remaining after outlier removal: {len(df)}")

Removed 263 outliers based on list_price
Removed 50 outliers based on sqft_living
Total records remaining after outlier removal: 2453


### 4. Encoding & Dropping Unnecessary Columns

In [94]:
# Drop identifier / high-cardinality / single-value columns
drop_cols = ['address', 'scraped_at', 'property_id', 'listing_id']
for col in ['state', 'city']:
    if col in df.columns and df[col].nunique() == 1:
        drop_cols.append(col)

df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# One-Hot Encoding for property_type
if 'property_type' in df.columns:
    df = pd.get_dummies(df, columns=['property_type'], drop_first=True, dtype=int)

df.head()

,listing_status,list_price,beds,baths,sqft_living,lot_sqft,year_built,city,zip_code,latitude,...,days_on_market,price_per_sqft,listing_type,parking_garage,stories,property_type_LAND,property_type_MOBILE,property_type_MULTI_FAMILY,property_type_SINGLE_FAMILY,property_type_TOWNHOMES
0,FOR_SALE,1674999,2.0,2.0,1587.0,131.0,2011.0,Austin,78701,30.265737,...,149.0,1055.0,for_sale,1.0,1.0,0,0,0,0,0
3,FOR_SALE,1495000,4.0,3.0,2057.0,144.0,2008.0,Austin,78701,30.267661,...,58.0,727.0,for_sale,4.0,1.0,0,0,0,0,0
4,FOR_SALE,495000,2.0,2.0,1169.0,301.0,1964.0,Austin,78701,30.280650,...,96.0,423.0,for_sale,1.0,1.0,0,0,0,0,0
5,FOR_SALE,560000,2.0,2.0,791.0,61.0,2019.0,Austin,78701,30.255947,...,88.0,708.0,for_sale,1.0,1.0,0,0,0,0,0
6,FOR_SALE,264900,1.0,1.0,728.0,113.0,2007.0,Austin,78701,30.265427,...,38.0,364.0,for_sale,1.0,1.0,0,0,0,0,0


### Feature Engineering

In [95]:
df["house_age"] = 2026 - df["year_built"]

df['baths_per_bed'] = df['baths'] / df['beds'].replace(0, 1)

df

,listing_status,list_price,beds,baths,sqft_living,lot_sqft,year_built,city,zip_code,latitude,...,listing_type,parking_garage,stories,property_type_LAND,property_type_MOBILE,property_type_MULTI_FAMILY,property_type_SINGLE_FAMILY,property_type_TOWNHOMES,house_age,baths_per_bed
0,FOR_SALE,1674999,2.0,2.0,1587.0,131.0,2011.0,Austin,78701,30.265737,...,for_sale,1.0,1.0,0,0,0,0,0,15.0,1.000000
3,FOR_SALE,1495000,4.0,3.0,2057.0,144.0,2008.0,Austin,78701,30.267661,...,for_sale,4.0,1.0,0,0,0,0,0,18.0,0.750000
4,FOR_SALE,495000,2.0,2.0,1169.0,301.0,1964.0,Austin,78701,30.280650,...,for_sale,1.0,1.0,0,0,0,0,0,62.0,1.000000
5,FOR_SALE,560000,2.0,2.0,791.0,61.0,2019.0,Austin,78701,30.255947,...,for_sale,1.0,1.0,0,0,0,0,0,7.0,1.000000
6,FOR_SALE,264900,1.0,1.0,728.0,113.0,2007.0,Austin,78701,30.265427,...,for_sale,1.0,1.0,0,0,0,0,0,19.0,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2761,SOLD,500000,3.0,2.0,1912.0,8032.0,2003.0,Austin,78735,30.253977,...,sold,2.0,2.0,0,0,0,0,0,23.0,0.666667
2762,SOLD,675000,2.0,2.0,1816.0,7048.0,2015.0,Austin,78735,30.257886,...,sold,2.0,1.0,0,0,0,0,0,11.0,1.000000
2763,SOLD,825000,3.0,2.0,1565.0,7048.0,1973.0,Austin,78735,30.243595,...,sold,2.0,1.0,0,0,0,1,0,53.0,0.666667
2764,SOLD,750000,3.0,2.0,2756.0,15037.0,2000.0,Austin,78735,30.243322,...,sold,2.0,2.0,0,0,0,1,0,26.0,0.666667


In [96]:
# correlation analysis
num_features = df.select_dtypes(include=['number'])
correlation_matrix = num_features.corr()
list_price_correlations = correlation_matrix['list_price'].abs().sort_values(ascending=False)
percent_list_price_correlations = list_price_correlations * 100
percent_list_price_correlations

list_price                     100.000000
sqft_living                     61.893710
baths                           45.251897
beds                            43.056780
price_per_sqft                  42.858102
property_type_SINGLE_FAMILY     30.387473
stories                         20.951603
zip_code                        17.410302
property_type_MOBILE            12.021943
longitude                       10.131307
baths_per_bed                    8.826080
latitude                         6.943585
days_on_market                   6.634995
property_type_TOWNHOMES          3.194860
property_type_MULTI_FAMILY       3.020494
property_type_LAND               2.287273
lot_sqft                         2.112788
year_built                       1.874399
house_age                        1.874399
hoa_fee                          1.401871
parking_garage                   1.007013
Name: list_price, dtype: float64

In [97]:
df.to_csv('../data/processed/processed_housing_data.csv', index=False)
print("Saved processed data successfully!")

Saved processed data successfully!
